# AQUAVOLTA — Cleaning of the 2023 workbook

Sheets `EXPORT`, `LOCAL`, `DEPENSES`, `SALAIRES` of `DATA FORMATION BI 2023.xlsx`.

**How to run** — install the dependencies once (`pip install -r analytics/requirements.txt`), place the confidential workbook in `data/raw/` (or set `AQUAVOLTA_DATA_DIR`), then run all cells from top to bottom. Each section reloads its own sheet into `res`, so sections can also be run independently after the two setup cells.

The source data is confidential: this notebook is published without outputs. See `analytics/README.md` and `analytics/calculation-audit.md`.

## Setup

In [ ]:
import pandas as pd
import missingno as msno
from datetime import datetime
import re

In [ ]:
# Paths, loading and shared financial helpers (see aquavolta_helpers.py)
from aquavolta_helpers import (
    DATA_DIR,
    OUTPUT_DIR,
    check_collection_identity,
    collection_ratio,
    impute_collection_from_client_ratio,
    load_sheet,
    save_output,
)

print("Source workbooks :", DATA_DIR)
print("Cleaned outputs  :", OUTPUT_DIR)

## Sheet `EXPORT`

Source: `DATA FORMATION BI 2023.xlsx` (header on row 5).

In [ ]:
res = load_sheet("DATA FORMATION BI 2023.xlsx", sheet_name='EXPORT', header=4)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

### Missing export collections — client-ratio imputation (corrected)

Two export invoices have no collected amount. The original approach estimates it from the average collection behaviour of the same client. That methodology is kept, but its implementation was corrected during the calculation audit (`analytics/calculation-audit.md`, finding F1):

| | Original | Corrected |
|---|---|---|
| Ratio unit | percentage (`× 100`) multiplied directly by revenue | fraction (0–1) |
| Row selection | fixed positions `res.loc[11]`, `res.loc[28]` and hard-coded client names | every row where the collection is missing, matched to its own client |
| Division | unprotected | zero or missing revenue gives `NaN` |

`comport` below is kept in **percent** for display only.

In [ ]:
# Collection behaviour per invoice, in PERCENT (exploration only - not used for imputation)
res["comport"] = collection_ratio(res['Recouvrement €'], res['CA €']) * 100
print(res['comport'])

In [ ]:
res['comport'].mean()

In [ ]:
# Behaviour of the clients that have at least one missing collection
missing_mask = res['Recouvrement €'].isna()
clients_cibles = sorted(res.loc[missing_mask, 'Nom du client'].dropna().unique())
res_filtré = res[res['Nom du client'].isin(clients_cibles)].copy()
print(res_filtré[['Nom du client', 'comport']])
print(res_filtré.groupby('Nom du client')['comport'].mean())

In [ ]:
# ORIGINAL IMPLEMENTATION - kept for traceability, NOT executed:
#
#   r1 = res_filtré['comport'].mean(skipna=True)                  # r1 is a PERCENTAGE (e.g. 92.3)
#   res.loc[11, "Recouvrement €"] = res.loc[11, 'CA €'] * r1      # revenue x 92.3 instead of x 0.923
#   r2 = res_filtré['comport'].mean(skipna=True)
#   res.loc[28, "Recouvrement €"] = res.loc[28, 'CA €'] * r2
#
# CORRECTED IMPLEMENTATION - same methodology, ratio used as a fraction,
# rows selected by condition instead of position:
res, audit_imputation = impute_collection_from_client_ratio(
    res,
    client_col='Nom du client',
    revenue_col='CA €',
    collected_col='Recouvrement €',
)
print(audit_imputation)

**Consistency control.** For every invoice, `CA € = Recouvrement € + Restant €` should hold. Rows listed below break that identity. If an imputed row appears here with `Restant € = CA €`, the source workbook records the invoice as entirely unpaid, and the business must decide whether the collection should be `0` rather than an estimate (audit finding F2). No value is changed by this cell.

In [ ]:
ecarts = check_collection_identity(res, 'CA €', 'Recouvrement €', 'Restant €')
print(f"{len(ecarts)} row(s) where CA € != Recouvrement € + Restant €")
print(ecarts)

In [ ]:
print(res.info())

In [ ]:
res["comport"]=(res['Recouvrement €']/res['CA €'])*100
print(res['comport'])

In [ ]:
def fix_date(date_str):
    if pd.isna(date_str):
        return None

    d = str(date_str).strip()
    d = d.replace("  ", " ")   # supprimer espaces doubles

    # ✅ Cas spécial : "YYYY-MM-DD 00:00:00"
    if re.match(r"^\d{4}-\d{2}-\d{2} 00:00:00$", d):
        dt = datetime.strptime(d, "%Y-%m-%d %H:%M:%S")
        return dt.strftime("%d/%m/%Y")

    # si contient heure → on garde juste date
    if re.search(r"\d{2}:\d{2}:\d{2}", d):
        d = d.split(" ")[0]

    # séparer jour/mois/année
    parts = re.split(r"[-/]", d)
    if len(parts) != 3:
        return d   # garder comme il est

    day, month, year = parts

    # corriger année mal écrite
    year = year.replace(" ", "")
    if len(year) == 2:  
        year = "20" + year
    elif len(year) != 4:
        return d   # garder comme il est

    try:
        day = int(day)
        month = int(month)
        year = int(year)
    except ValueError:
        return d   # garder comme il هو

    # corriger mois hors limite
    if month > 12:
        month = 12
    if month < 1:
        month = 1

    # max jours par mois
    max_days = {
        1: 31, 2: 29 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 28,
        3: 31, 4: 30, 5: 31, 6: 30,
        7: 31, 8: 31, 9: 30, 10: 31,
        11: 30, 12: 31
    }

    if day > max_days[month]:
        day = max_days[month]
    if day < 1:
        day = 1

    fixed_date = datetime(year, month, day)
    return fixed_date.strftime("%d/%m/%Y")

# 🔹 Appliquer sur ton DataFrame
res["DATE"] = res["DATE"].apply(fix_date)
print(res)

In [ ]:
save_output(res, "OUTOUT___EXPORT2023.xlsx")

## Sheet `LOCAL`

Source: `DATA FORMATION BI 2023.xlsx` (header on row 5).

In [ ]:
res = load_sheet("DATA FORMATION BI 2023.xlsx", sheet_name='LOCAL', header=4)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
res=res.rename(columns={"date":"type"})
print(res)

> **Review note (audit finding F3).** Missing client names are replaced by the most frequent client. This attributes unidentified invoices to one real client and inflates its revenue in client-level KPIs. Calculation left unchanged; a neutral label such as `CLIENT NON IDENTIFIE` is recommended.

In [ ]:
mode=res[['NOM DU CLIENT']].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

> **Review note (audit finding F5).** The next two cells overwrite the VAT and TTC of *every* local invoice with a 19 % rate, including invoices whose source VAT was zero, and `RESTANT` is not recomputed. Calculation left unchanged; to be confirmed with the business.

In [ ]:
res['tva 19%']=res['HT']*0.19
print(res['tva 19%'])

In [ ]:
res['TTC']=res['HT']*1.19
print(res['TTC'])

In [ ]:
print(res.describe())

In [ ]:
res["Date"] = pd.to_datetime(res["Date"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
save_output(res, "OUTPUT___LOCAL2023.xlsx")

## Sheet `DEPENSES`

Source: `DATA FORMATION BI 2023.xlsx` (header on row 7).

In [ ]:
res = load_sheet("DATA FORMATION BI 2023.xlsx", sheet_name='DEPENSES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
res['TTC']=res['HT']+res['TVA']
print(res['TTC'])

In [ ]:
print(res.info())

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
save_output(res, "OUTPUT___DEPENSE2023.xlsx")

## Sheet `SALAIRES`

Source: `DATA FORMATION BI 2023.xlsx` (header on row 7).

In [ ]:
res = load_sheet("DATA FORMATION BI 2023.xlsx", sheet_name='SALAIRES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
count_duplicates = res.duplicated().sum()

print(count_duplicates)

In [ ]:
mois_map = {
    "JANVIER": "01",
    "FEVRIER": "02",
    "MARS": "03",
    "AVRIL": "04",
    "MAI": "05",
    "JUIN": "06",
    "JUILLET": "07",
    "AOUT": "08",
    "SEPTEMBRE": "09",
    "OCTOBRE": "10",
    "NOVEMBRE": "11",
    "DECEMBRE": "12"
}
res["date"] = res["MOIS"].map(mois_map)  
res["date"] = "01/" + res["date"] + "/2023"
print(res[["MOIS", "date"]].head())

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
save_output(res, "OUTPUT____SALAIRE2023.xlsx")